# Canonicalization of Guitar Fretboards from Video Frames
This notebook is a demo of extrscting a canonical, top-down view of a guitar fretboard from raw video frames. This is an improvement on a previous demo that used only a Mask R-CNN seg mask. The original method worked fairly well, but struggled when the mask was noisy or partially occluded by hands.

**Next steps:** switch to YOLO?

In [1]:
from __future__ import annotations

import io
import os
from pathlib import Path

import numpy as np
import torch
import torchvision
import torchvision.transforms.functional as F
from PIL import Image
import cv2
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import ipywidgets as widgets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using {device}")

def find_file_upwards(filename: str, start: Path | None = None, max_depth: int = 6) -> Path | None:
    """Search `start` and its parents for `filename`. Returns the first match."""
    start = start or Path.cwd()
    start = start.resolve()
    for i, p in enumerate([start, *start.parents]):
        if i > max_depth:
            break
        candidate = p / filename
        if candidate.exists():
            return candidate
    return None


using cpu


In [2]:

def get_model_instance_segmentation(num_classes: int):
    """Mask R-CNN ResNet-50 FPN with a custom head for `num_classes`."""
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )
    return model


def load_fretboard_model(weights_path: Path, num_classes: int, device: torch.device):
    model = get_model_instance_segmentation(num_classes=num_classes)
    state = torch.load(str(weights_path), map_location=device)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model.load_state_dict(state, strict=False)
    model.to(device)
    model.eval()
    return model


@torch.no_grad()
def get_fretboard_mask_from_image(
    model,
    img_pil: Image.Image,
    device: torch.device,
    confidence_threshold: float = 0.5,
) -> np.ndarray:
    """Run the model, return a binary mask (H,W) in {0,255}."""
    img = F.to_tensor(img_pil).to(device)
    out = model([img])[0]

    if len(out.get("scores", [])) == 0:
        return np.zeros((img_pil.height, img_pil.width), dtype=np.uint8)

    scores = out["scores"].detach().cpu().numpy()
    keep = np.where(scores >= confidence_threshold)[0]
    if keep.size == 0:
        return np.zeros((img_pil.height, img_pil.width), dtype=np.uint8)

    best = keep[np.argmax(scores[keep])]
    mask = out["masks"][best, 0].detach().cpu().numpy()
    mask = (mask >= 0.5).astype(np.uint8) * 255
    return mask


def clean_mask(mask: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    m = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel, iterations=1)
    return m


def get_contour_and_min_area_rect(mask: np.ndarray):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None

    contour = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(contour)
    box = cv2.boxPoints(rect)  # 4x2
    box = box.astype(np.float32)
    return contour, box


def order_points(pts: np.ndarray) -> np.ndarray:
    """Return pts ordered as [top-left, top-right, bottom-right, bottom-left]."""
    pts = np.array(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)

    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.array([tl, tr, br, bl], dtype=np.float32)


def warp_to_canonical(img_bgr: np.ndarray, src_corners: np.ndarray, out_w: int, out_h: int):
    src = order_points(src_corners)
    dst = np.array([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]], dtype=np.float32)
    H = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(img_bgr, H, (out_w, out_h))
    return H, src, warped


In [3]:
# === User configuration ===
# These can be in the repo root or the same folder as this notebook.
MODEL_WEIGHTS_NAME = "model_weights.pt"
DEFAULT_IMAGE_NAME = "guitar_test_clean.png"

CONFIDENCE_THRESHOLD = 0.5
OUT_WIDTH, OUT_HEIGHT = 1024, 256

weights_path = find_file_upwards(MODEL_WEIGHTS_NAME)
if weights_path is None:
    raise FileNotFoundError(
        f"Weights not found: {MODEL_WEIGHTS_NAME}\n"
        f"Looked in: {Path.cwd().resolve()} and parents.\n"
        f"How to Fix: place {MODEL_WEIGHTS_NAME} in the same folder as the notebook (or repo root)."
    )

model = load_fretboard_model(weights_path, num_classes=2, device=device)
print(f"Model loaded from: {weights_path}")


Model loaded from: C:\Users\arnav\Python Notebooks\VIP Research\spring 26\canonical_demo\model_weights.pt


In [4]:
def run_full_pipeline(image_path: str):
    img_pil = Image.open(image_path).convert("RGB")
    img_bgr = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

    raw_mask = get_fretboard_mask_from_image(
        model, img_pil, device=device, confidence_threshold=CONFIDENCE_THRESHOLD
    )
    cleaned_mask = clean_mask(raw_mask, kernel_size=5)

    contour, box = get_contour_and_min_area_rect(cleaned_mask)
    if contour is None or box is None:
        print("No fretboard contour found.")
        return None

    # Visual overlay for sanity
    overlay = img_bgr.copy()
    cv2.drawContours(overlay, [box.astype(np.int32)], -1, (0, 255, 0), 3)

    H, src_corners, warped_bgr = warp_to_canonical(img_bgr, box, OUT_WIDTH, OUT_HEIGHT)

    # Display
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(cleaned_mask, cmap="gray")
    axes[1].set_title("Cleaned Mask")
    axes[1].axis("off")

    axes[2].imshow(cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2RGB))
    axes[2].set_title("Warped Canonical View")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    return {"homography": H, "src_corners": src_corners, "warped_bgr": warped_bgr, "mask": cleaned_mask}


# ui stuff (voila)
uploader = widgets.FileUpload(accept="image/*", multiple=False)
run_btn = widgets.Button(description="Run pipeline", button_style="primary")
out = widgets.Output()

display(widgets.HTML("<h3>Upload an image and run</h3>"))
display(widgets.HTML("Or, click run to use default image"))
display(uploader, run_btn, out)

def _save_uploaded_image() -> str | None:
    if not uploader.value:
        return None

    uploaded = uploader.value[0]
    name = uploaded.get("name", "uploaded.png")
    ext = Path(name).suffix.lower()
    if ext not in [".png", ".jpg", ".jpeg", ".bmp", ".webp"]:
        ext = ".png"

    path = Path(f"uploaded_image{ext}").resolve()
    with open(path, "wb") as f:
        f.write(uploaded["content"])
    return str(path)

def _run(_):
    with out:
        clear_output()

        img_path = _save_uploaded_image()
        if img_path is None:
            default_img = find_file_upwards(DEFAULT_IMAGE_NAME)
            if default_img is None:
                print("Please upload an image (no default image found).")
                return
            img_path = str(default_img)

        print(f"Using image: {img_path}")
        run_full_pipeline(img_path)

run_btn.on_click(_run)


HTML(value='<h3>Upload an image and run</h3>')

HTML(value='Or, click run to use default image')

FileUpload(value=(), accept='image/*', description='Upload')

Button(button_style='primary', description='Run pipeline', style=ButtonStyle())

Output()